# Evaluación Parcial 2 — Agente INGER (Groq + LangGraph + LangSmith)
**Propósito:** ejecutar un agente sobre registros y procedimientos **completamente ficticios**, con recuperación RAG, memoria, herramientas tipadas y seguimiento de trazas. Este notebook requiere subir el ZIP del proyecto e ingresar claves privadas para acceder a las API. **No cargar datos clínicos reales.**

## 1. Cargar el proyecto ZIP
Suba el archivo `INGER_EP2_Proyecto_Completo.zip`. No comparta claves ni datos de pacientes. El ZIP se descomprime en `/content/INGER_EP2`.

In [ ]:
from google.colab import files
import zipfile, io, os
subidos = files.upload()  # Subir INGER_EP2_Proyecto_Completo.zip
nombre_zip = next((n for n in subidos if n.lower().endswith('.zip')), None)
if not nombre_zip:
    raise ValueError('Debe subir un archivo .zip del proyecto.')
with zipfile.ZipFile(io.BytesIO(subidos[nombre_zip])) as z:
    z.extractall('/content/')
os.chdir('/content/INGER_EP2')
print('Proyecto preparado:', os.getcwd())

## 2. Instalar dependencias
Primera ejecución descarga bibliotecas y el modelo local de embeddings multilingüe. Se instala LangGraph, FAISS, Groq, LangSmith y Pydantic.

In [ ]:
%pip install -q -r requirements.txt

## 3. Conectar Groq y LangSmith sin exponer claves
En Colab utilice el icono de llave (Secrets) con los nombres `GROQ_API_KEY` y `LANGSMITH_API_KEY`; active acceso del notebook. Si no están, se solicitan con ingreso oculto. No muestre estas celdas durante el video.

In [ ]:
import os, getpass
from google.colab import userdata

def recuperar_secreto(nombre):
    try:
        v = userdata.get(nombre)
        if v: return v
    except Exception:
        pass
    return getpass.getpass(f'Ingrese {nombre} (no se mostrará): ')

os.environ['GROQ_API_KEY'] = recuperar_secreto('GROQ_API_KEY')
os.environ['LANGSMITH_API_KEY'] = recuperar_secreto('LANGSMITH_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_PROJECT'] = 'INGER-EP2-GROQ-DEMO'
os.environ['LANGGRAPH_STRICT_MSGPACK'] = 'true'
os.environ['GROQ_MODEL'] = 'openai/gpt-oss-20b'
print('Variables necesarias configuradas; claves ocultas.')

## 4. Ejecutar pruebas locales (sin consumo de API)
La ejecución de pruebas ayuda a distinguir la lógica local de las llamadas reales a Groq y LangSmith.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], text=True, capture_output=True)
print(r.stdout + r.stderr)
assert r.returncode == 0, 'Hay fallos en las pruebas locales.' 

## 5. Inicializar el agente funcional (con Groq real)
El agente crea embeddings locales de los textos ficticios, construye un índice FAISS, carga herramientas tipadas y utiliza LangGraph con checkpoints SQLite.

In [ ]:
from pathlib import Path
from inger_agent.grafo import AgenteINGER
agente = AgenteINGER(Path.cwd(), modo='groq')
print('LangGraph iniciado. Modelo Groq:', os.environ['GROQ_MODEL'])

## 6. Demostración de autonomía y memoria de corto plazo
Todas las consultas usan el mismo `hilo`, por lo cual LangGraph mantiene el estado entre invocaciones. Se muestra intención, plan, salida y fuentes.

In [ ]:
hilo = 'video-ep2-inger'
consultas = [
    'Resume los antecedentes de seguimiento del CASO-001',
    '¿Y sus pendientes explícitamente registrados?',
    '¿Qué casos requieren revisión administrativa según las reglas simuladas?',
    '¿Qué establece el procedimiento simulado sobre trazabilidad?'
]
for pregunta in consultas:
    resultado = agente.preguntar(pregunta, hilo=hilo, rol='gestor_demo')
    print('\nCONSULTA:', pregunta)
    print('RUTA:', resultado['intencion'])
    print('PLAN:', ' → '.join(resultado['plan']))
    print('RESPUESTA:\n', resultado['respuesta'])

## 7. Escritura controlada y adaptación a cambios
Se prueba primero un usuario solo-lectura y luego un usuario autorizado a crear **borradores ficticios**; los Excel originales nunca se modifican.

In [ ]:
pregunta = 'Guarda borrador de seguimiento para CASO-001'
denegado = agente.preguntar(pregunta, hilo=hilo, rol='lector_demo')
print('ROL LECTOR / DENEGADO:', denegado['escritura'])
autorizado = agente.preguntar(pregunta, hilo=hilo, rol='gestor_demo')
print('ROL GESTOR / PERMITIDO:', autorizado['escritura'])
print('Borradores locales:', agente.memoria.borradores(hilo))

## 8. Memoria a largo plazo y recuperación
Los eventos se guardan en SQLite. Aquí revisamos solo metadatos ficticios y consultamos el estado conversacional por hilo.

In [ ]:
eventos = agente.memoria.recuerdos(hilo)
print('Eventos persistidos:', len(eventos))
for evento in eventos[:4]: print(evento['caso_id'], evento['intencion'], evento['creado'])
estado = agente.grafo.get_state({'configurable': {'thread_id': hilo}})
print('Mensajes guardados en checkpoints:', len(estado.values.get('messages', [])))

## 9. Observabilidad real con LangSmith
Abra [LangSmith](https://smith.langchain.com/), seleccione proyecto `INGER-EP2-GROQ-DEMO` y una traza reciente. Deben aparecer nodos del grafo, llamadas a herramientas y generación del modelo. Se consulta el listado de ejecuciones sin imprimir contenido sensible.

In [ ]:
from langsmith import Client
cliente = Client()
try:
    ejecuciones = list(cliente.list_runs(project_name=os.environ['LANGSMITH_PROJECT'], limit=6))
    print('Trazas disponibles en LangSmith:', len(ejecuciones))
    for run in ejecuciones:
        print(run.name, run.run_type, str(run.id)[:8])
except Exception as e:
    print('No se pudo verificar LangSmith. Revise clave/proyecto/red:', type(e).__name__, str(e)[:180])

## 10. Cerrar
El video debe mostrar las llamadas reales y el panel LangSmith, sin revelar claves. No se garantiza una traza hasta que la ejecución autenticada aparezca en LangSmith.

In [ ]:
agente.cerrar()
print('Sesión de demostración cerrada.')